## Load & visualise the saved FENE-v2 HDF5 dataset

Generated with:
```
pixi run python data_generation/fene/generate.py -n 10 -r 0
```
Default parameters: $\varepsilon \approx 0.03$, $c = 1$, $T = 1.0$, `save_every=20` → ~51 snapshots per trajectory.

Stored variable: `u_sol_all` — the slow field $u(\xi, \tau) = r_n / \varepsilon^2$ in the co-moving frame,
shape `(N, n_frames, N_kdv, 1)` — trailing channel dim squeezed for plotting.

In [ ]:
import h5py
import numpy as np
import rootutils
from IPython.display import HTML
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
root = rootutils.setup_root(".", indicator=".project-root", pythonpath=True)
HDF5_PATH = root / "data/fene_v2/custom_v1_fene_v2_nx256_nchain209_num10_seed0.hdf5"

with h5py.File(HDF5_PATH, "r") as f:
    t_coord = f["t_coord"][:]  # (n_frames,)  slow time
    x_coord = f["x_coord"][:]  # (N_kdv,)     co-moving xi
    u_sol_all = f["u_sol_all"][:]  # (N, n_frames, N_kdv, 1)
    coef_ks = f["coef/u_ic/ks_all"][:]
    coef_amps = f["coef/u_ic/amps_all"][:]
    coef_phases = f["coef/u_ic/phases_all"][:]
    c = float(f["pde_info/c"][()])
    eps = float(f["pde_info/eps"][()])
    H = float(f["pde_info/H"][()])
    R = float(f["pde_info/R"][()])

    print("Datasets in file:")

    def _print_item(name, obj):
        if hasattr(obj, "shape"):
            if obj.shape == ():
                print(f"  {name}: {obj.shape} = {obj[()]}")
            else:
                print(f"  {name}: {obj.shape}")
        else:
            print(f"  {name}")

    f.visititems(_print_item)

# squeeze channel dim: (N, n_frames, N_kdv)
u_sol_all = u_sol_all[..., 0]

print(f"\nu_sol_all : {u_sol_all.shape}  (samples, frames, N_kdv)")
print(f"t_coord   : {t_coord.shape}  -> tau in [{t_coord[0]:.4f}, {t_coord[-1]:.4f}]")
print(f"x_coord   : {x_coord.shape}  -> xi in [{x_coord[0]:.4f}, {x_coord[-1]:.4f}]")
print(f"c={c}, eps={eps:.5f}, H={H}, R={R:.6f}")

In [ ]:
# ── Static overview: all samples at tau=0, tau_mid, tau_end ───────────────────
n_samples = u_sol_all.shape[0]
n_frames = u_sol_all.shape[1] - 1
snap_idx = [0, n_frames // 2, n_frames]
snap_t = t_coord[snap_idx]

fig, axes = plt.subplots(n_samples, 3, figsize=(13, 2.2 * n_samples), sharey="row")
for i in range(n_samples):
    for col, (si, t) in enumerate(zip(snap_idx, snap_t)):
        axes[i, col].plot(x_coord, u_sol_all[i, si], lw=1.2)
        axes[i, col].set_xlim(x_coord[0], x_coord[-1])
        if i == 0:
            axes[i, col].set_title(rf"$\tau$ = {t:.4f}")
        if col == 0:
            axes[i, col].set_ylabel(f"sample {i}")

fig.suptitle(r"FENE-v2 slow field $u(\xi,\tau)$ snapshots for all samples", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── Animated lines: all samples evolving together ─────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
cmap = plt.get_cmap("tab10")
lines = [ax.plot(x_coord, u_sol_all[i, 0], lw=1.2, color=cmap(i % 10), label=f"s{i}")[0] for i in range(n_samples)]
title = ax.set_title(rf"$\tau$ = {t_coord[0]:.4f}")
ax.set_xlabel(r"$\xi$")
ax.set_ylabel(r"$u(\xi,\tau)$")
ax.set_xlim(x_coord[0], x_coord[-1])
u_abs_max = float(np.abs(u_sol_all).max())
ax.set_ylim(-u_abs_max * 1.1, u_abs_max * 1.1)
ax.legend(loc="upper right", fontsize=7, ncol=2)
fig.tight_layout()


def _update(frame):
    for i, ln in enumerate(lines):
        ln.set_ydata(u_sol_all[i, frame])
    title.set_text(rf"$\tau$ = {t_coord[frame]:.4f}")
    return lines + [title]


ani = FuncAnimation(fig, _update, frames=u_sol_all.shape[1], interval=80, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
# ── Space-time heatmap for sample 0 ──────────────────────────────────────────
i = 0
fig, ax = plt.subplots(figsize=(8, 5))
vmax = float(np.max(np.abs(u_sol_all[i])))
im = ax.imshow(
    u_sol_all[i],
    aspect="auto",
    origin="lower",
    extent=[x_coord[0], x_coord[-1], t_coord[0], t_coord[-1]],
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
ax.set_xlabel(r"$\xi$")
ax.set_ylabel(r"$\tau$")
ax.set_title(rf"FENE-v2: $u(\xi,\tau)$ — sample {i}")
fig.colorbar(im, ax=ax, label=r"$u$")
fig.tight_layout()
plt.show()

In [ ]:
# ── RMS amplitude per sample over time ────────────────────────────────────────
u_rms = np.sqrt(np.mean(u_sol_all**2, axis=2))  # (n_samples, n_frames)

fig, ax = plt.subplots(figsize=(8, 3))
for i in range(n_samples):
    ax.plot(t_coord, u_rms[i], lw=1.0, alpha=0.7, color=cmap(i % 10), label=f"s{i}")
ax.set_xlabel(r"$\tau$")
ax.set_ylabel(r"$\mathrm{rms}(u)$")
ax.set_title("RMS slow-field amplitude over time (all samples)")
ax.legend(loc="upper right", fontsize=7, ncol=2)
fig.tight_layout()
plt.show()